### Importing Libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import tensorflow.keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Dense, Input, LSTM, Embedding, Dropout, Activation,Bidirectional
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential

### Handling Pre-processed data

In [6]:
data = pd.read_csv('../preprocess_data.csv')
data.drop(['task_2','Unnamed: 0', 'Unnamed: 0.1','text'], axis=1, inplace=True)
data.head()

,_id,task_1,text_clean
0,60c5d6bf5659ea5e55defa2c,HOF,made amp amp onli abl start make money sustain...
1,60c5d6bf5659ea5e55def461,HOF,technic still turn back clock dick head
2,60c5d6bf5659ea5e55defaad,NOT,govt stop think world media liber gang ani opt...
3,60c5d6bf5659ea5e55def419,HOF,soldier japan dick head
4,60c5d6bf5659ea5e55def7fa,HOF,would better ask think sleazi shitbag lmao


In [7]:
sentences = data['text_clean'].astype(str)
tokenizer = Tokenizer(num_words = 1500,split=' ')
tokenizer.fit_on_texts(sentences)
sequence = tokenizer.texts_to_sequences(sentences)

In [8]:

max_seq_len = 2500

index_of_words = tokenizer.word_index
print("No of unique words : ", len(index_of_words))

X = pad_sequences(sequence, maxlen = max_seq_len)
Y = data['task_1']

print(X)

No of unique words :  8255
[[   0    0    0 ...  170    3  210]
 [   0    0    0 ...   72   54   73]
 [   0    0    0 ...    3   52   13]
 ...
 [   0    0    0 ...  817   45  156]
 [   0    0    0 ...  213   99   38]
 [   0    0    0 ... 1166  236   57]]


In [9]:
embed_dim = 256
vocabSize = len(index_of_words)
lstm_out = 64

In [10]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.15, random_state = 0)
Y_true = Y_test
Y_train = pd.get_dummies(Y_train).values
Y_test = pd.get_dummies(Y_test).values

In [11]:
model = Sequential()
model.add(Embedding(vocabSize, embed_dim,input_length = 2500))
model.add(LSTM(lstm_out, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(2, activation='softmax'))
model.compile(loss = 'binary_crossentropy', optimizer='adam',metrics = ['accuracy'])
print(model.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 2500, 256)         2113280   
                                                                 
 lstm (LSTM)                 (None, 64)                82176     
                                                                 
 dense (Dense)               (None, 2)                 130       
                                                                 
Total params: 2195586 (8.38 MB)
Trainable params: 2195586 (8.38 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [12]:
from keras.callbacks import ModelCheckpoint
checkpoint = ModelCheckpoint("hasoc_a_2.h5", monitor='val_loss', verbose=1, save_best_only=True,
save_weights_only=False, mode='auto')

In [13]:
model.fit(X_train, Y_train, batch_size = 32, epochs = 5, validation_data=(X_test, Y_test) , callbacks=[checkpoint])

Epoch 1/5
103/103 [==============================] - ETA: 0s - loss: 0.5797 - accuracy: 0.7002
Epoch 1: val_loss improved from inf to 0.49202, saving model to hasoc_a_2.h5
103/103 [==============================] - 660s 6s/step - loss: 0.5797 - accuracy: 0.7002 - val_loss: 0.4920 - val_accuracy: 0.7764
Epoch 2/5


C:\Users\8888\Anaconda3\envs\pythonProject11\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


103/103 [==============================] - ETA: 0s - loss: 0.4110 - accuracy: 0.8187
Epoch 2: val_loss did not improve from 0.49202
103/103 [==============================] - 670s 7s/step - loss: 0.4110 - accuracy: 0.8187 - val_loss: 0.4970 - val_accuracy: 0.7851
Epoch 3/5
103/103 [==============================] - ETA: 0s - loss: 0.3263 - accuracy: 0.8631
Epoch 3: val_loss did not improve from 0.49202
103/103 [==============================] - 666s 6s/step - loss: 0.3263 - accuracy: 0.8631 - val_loss: 0.5306 - val_accuracy: 0.7522
Epoch 4/5
103/103 [==============================] - ETA: 0s - loss: 0.2882 - accuracy: 0.8870
Epoch 4: val_loss did not improve from 0.49202
103/103 [==============================] - 560s 5s/step - loss: 0.2882 - accuracy: 0.8870 - val_loss: 0.5908 - val_accuracy: 0.7591
Epoch 5/5
103/103 [==============================] - ETA: 0s - loss: 0.2182 - accuracy: 0.9158
Epoch 5: val_loss did not improve from 0.49202
103/103 [==============================] - 545

In [14]:
model.load_weights('hasoc_a_2.h5')
model.evaluate(X_test,Y_test)

19/19 [==============================] - 16s 852ms/step - loss: 0.4920 - accuracy: 0.7764


[0.492023229598999, 0.7764298319816589]

In [15]:
Y_pred = model.predict(X_test)

19/19 [==============================] - 16s 854ms/step


In [16]:
y_actual = []
for i in Y_true:
    if i =='NOT':
        y_actual.append(1)
    else :
        y_actual.append(0)

pred_class = []
for i in Y_pred:
    pred_class.append(np.argmax(i))

In [17]:
print(classification_report(y_actual , pred_class))

              precision    recall  f1-score   support

           0       0.80      0.89      0.84       380
           1       0.72      0.56      0.63       197

    accuracy                           0.78       577
   macro avg       0.76      0.73      0.74       577
weighted avg       0.77      0.78      0.77       577

